# RT-DETR 공통 COCO 평가
팀원이 각자 만든 모델로 **고정된 평가 이미지 1,000장**을 추론한 뒤, COCO 형식 예측 JSON을 업로드합니다. 이 노트북은 데이터를 준비하고 mAP50:95·AP50·AP75·클래스별 AP를 계산합니다. **모델 추론이나 양자화는 실행하지 않습니다.**

In [ ]:
import os, pathlib, subprocess, sys
repo = pathlib.Path('/content/haiteam4_finetune')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/foxyaef/haiteam4_finetune.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, 'prepare_data.py'], check=True)
print('Protocol:', (repo / 'protocol.json').read_text(encoding='utf-8'))

## 예측 JSON 업로드 및 채점
파일은 COCO 탐지 결과 객체의 배열이어야 합니다. 각 객체에는 `image_id`, 원래 COCO `category_id`, 원본 이미지 픽셀 기준 `bbox=[x,y,width,height]`, `score`가 필요합니다. 예측 이미지 ID가 고정 평가 목록에 없으면 중단합니다.

In [ ]:
from google.colab import files
import datetime, zipfile
upload_dir = repo / 'submissions'
upload_dir.mkdir(exist_ok=True)
os.chdir(upload_dir)
uploaded = files.upload()
os.chdir(repo)
if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith('.json'):
    raise ValueError('COCO 예측 JSON 파일 하나를 올리세요')
prediction_file = upload_dir / next(iter(uploaded))
del uploaded
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S')
result_dir = repo / 'results' / ('evaluation_' + stamp)
subprocess.run([sys.executable, 'scripts/evaluate_predictions.py', '--predictions', str(prediction_file),
                '--output', str(result_dir)], check=True)
bundle = repo / 'results' / ('evaluation_' + stamp + '.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ('metrics.json', 'class_metrics.csv', 'provenance.json'):
        archive.write(result_dir / name, arcname=name)
files.download(str(bundle))